# HazeCast SG — Pipeline Walkthrough

Runs the same 5 steps as `README.md`'s "Running it" section, but interactively,
one stage at a time, so you can inspect intermediate output (row counts, date
ranges, gap warnings, evaluation tables) as you go.

**Before running:** place the raw source files under `data/raw/{pm25,firms,weather}/`
— see `data/README.md` for exact filenames and where to get them. This notebook
doesn't re-explain that step; it assumes the files are already there.

If you're running this via `docker compose up jupyter` (see `docker-compose.yml`),
the repo root is mounted at `/app` and `hazecast` is already importable. If you're
running JupyterLab locally instead (`pip install -r requirements.txt && jupyter lab`),
run the cell below first — it adds `src/` to the path so the same imports work
either way.

In [1]:
import sys, os, logging
from pathlib import Path

# Makes `import hazecast...` work whether this notebook is run from inside
# the Docker container (PYTHONPATH already set, see docker/pipeline.Dockerfile)
# or from a local `jupyter lab` with no PYTHONPATH set at all.
repo_root = Path.cwd()
if not (repo_root / "src" / "hazecast").exists():
    # notebook launched from notebooks/ instead of the repo root
    repo_root = repo_root.parent
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Jupyter sets the kernel's cwd to wherever the .ipynb file lives (this
# notebook's own folder, when opened via JupyterLab's file browser) --
# NOT the repo root. That silently breaks anything that writes to a
# relative path, most importantly MLflow's default local artifact store
# (`./mlruns/`), which would otherwise land inside notebooks/ instead of
# next to data/ and mlflow.db. Pin cwd to the repo root explicitly so
# every relative path in every stage below resolves the same way it does
# when running `python -m hazecast....` from the repo root.
os.chdir(repo_root)

logging.basicConfig(level=logging.INFO, force=True)

import hazecast
print("hazecast package OK, repo root:", repo_root)

hazecast package OK, repo root: /app


## 1. Ingest raw sources → bronze

One cell per source. Each prints the row count and date range it found — compare these against `data/README.md`'s expected ranges as a first sanity check.

In [2]:
from hazecast.ingest.pm25 import load_pm25_bronze, write_bronze as write_pm25_bronze

pm25_bronze = load_pm25_bronze()
write_pm25_bronze(pm25_bronze)
print(f"pm25: {len(pm25_bronze):,} rows, {pm25_bronze['ts'].min()} to {pm25_bronze['ts'].max()}")

pm25: 110,012 rows, 2014-04-01 01:00:00 to 2026-10-18 23:00:00


In [3]:
from hazecast.ingest.firms import load_firms_bronze, write_bronze as write_firms_bronze

firms_bronze = load_firms_bronze()
write_firms_bronze(firms_bronze)
print(f"firms: {len(firms_bronze):,} detections, {firms_bronze['date'].min()} to {firms_bronze['date'].max()}")
print(firms_bronze['region'].value_counts())

INFO:hazecast.ingest.firms:Dropped 301783 exact-duplicate detections


firms: 2,291,292 detections, 2014-01-01 to 2026-10-16
region
kalimantan    1336134
sumatra        955158
Name: count, dtype: int64


In [4]:
from hazecast.ingest.weather import load_weather_bronze, write_bronze as write_weather_bronze

weather_bronze = load_weather_bronze()
write_weather_bronze(weather_bronze)
for point, g in weather_bronze.groupby("point"):
    print(f"{point}: {len(g):,} rows, {g['ts'].min()} to {g['ts'].max()}")

palangka: 109,968 rows, 2014-04-01 00:00:00 to 2026-10-16 23:00:00
palembang: 109,968 rows, 2014-04-01 00:00:00 to 2026-10-16 23:00:00
pontianak: 109,968 rows, 2014-04-01 00:00:00 to 2026-10-16 23:00:00
riau: 109,968 rows, 2014-04-01 00:00:00 to 2026-10-16 23:00:00
singapore: 109,968 rows, 2014-04-01 00:00:00 to 2026-10-16 23:00:00


## 2. Bronze → Silver

Aligns each source to a full hourly (or daily, for fire counts) grid and flags gaps — this is where the pipeline surfaces data-quality issues rather than hiding them.

In [5]:
from hazecast.bronze_to_silver import pm25_silver, weather_silver, firms_daily_silver, write_silver

pm25_s = pm25_silver()
write_silver(pm25_s, "pm25")
print(f"silver/pm25: {len(pm25_s):,} rows")

weather_s = weather_silver()
write_silver(weather_s, "weather")
print(f"silver/weather: {len(weather_s):,} rows")

firms_daily_s = firms_daily_silver()
write_silver(firms_daily_s, "firms_daily")
print(f"silver/firms_daily: {len(firms_daily_s):,} days, {int(firms_daily_s['data_gap'].sum())} flagged as likely data gaps")

silver/pm25: 110,015 rows
silver/weather: 549,840 rows
silver/firms_daily: 4,672 days, 73 flagged as likely data gaps


## 3. Silver → Gold

Builds the one-row-per-day feature + label table. See `silver_to_gold.py`'s docstring and `tests/test_pipeline_smoke.py` for how leak-freedom is enforced and tested.

In [6]:
from hazecast.silver_to_gold import build_gold, write_gold

gold = build_gold(pm25_s, weather_s, firms_daily_s)
write_gold(gold)
print(f"gold: {gold.shape[0]:,} rows x {gold.shape[1]} columns, {gold['date'].min()} to {gold['date'].max()}")
gold.tail()

gold: 4,584 rows x 64 columns, 2014-04-01 00:00:00 to 2026-10-18 00:00:00


,date,fire_sumatra_lag0,fire_sumatra_lag1,fire_sumatra_lag2,fire_sumatra_lag3,fire_sumatra_roll3,fire_sumatra_roll7,fire_sumatra_log_lag0,fire_kalimantan_lag0,fire_kalimantan_lag1,...,today_west,today_central,today_national,label_north,label_south,label_east,label_west,label_central,label_national,label_low_coverage
4579,2026-10-14,2357.0,727.0,800.0,1269.0,3884.0,9476.0,7.765569,4336.0,3142.0,...,154.750000,136.041667,121.800000,83.541667,116.875000,125.125,127.166667,148.958333,120.333333,False
4580,2026-10-15,1566.0,2357.0,727.0,800.0,4650.0,8906.0,7.356918,1905.0,4336.0,...,127.166667,148.958333,120.333333,104.583333,127.833333,138.625,144.541667,161.375000,135.391667,False
4581,2026-10-16,432.0,1566.0,2357.0,727.0,4355.0,7991.0,6.070738,88.0,1905.0,...,144.541667,161.375000,135.391667,82.541667,117.375000,133.000,161.625000,136.041667,126.116667,False
4582,2026-10-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,161.625000,136.041667,126.116667,58.166667,72.041667,76.625,118.166667,83.083333,81.616667,False
4583,2026-10-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,118.166667,83.083333,81.616667,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Train — leave-one-season-out evaluation

Compares 4 candidates per region, holding out one full haze season at a time (see `hazecast.config.HAZE_SEASONS`): `persistence`, `linreg`, `xgboost` (plain MAE-minimizing regression), and `xgboost_q85` (quantile regression targeting the 85th percentile — built specifically to fix `xgboost`'s tendency to under-predict rare elevated days). Read `README.md`'s **Results** section for the full writeup of why `xgboost_q85` exists and how `XGB_QUANTILE_ALPHA` was chosen — in short, MAE alone hides how each model does on elevated days specifically, which is why `recall_elevated` / `precision_elevated` are broken out below too.

In [18]:
#old

from hazecast.train import evaluate_all, train_final_models
from hazecast.config import MODELS_DIR

eval_df = evaluate_all(gold)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
eval_df.to_csv(MODELS_DIR / "eval_summary.csv", index=False)

pivot = eval_df.pivot_table(index=["target", "season"], columns="model", values="mae")
pivot.round(2)

INFO:hazecast.train:Evaluating leave-one-season-out on: ['2015', '2019']


model            linreg  persistence  xgboost  xgboost_q85
target   season                                           
central  2015     14.16        15.76    13.84        15.70
         2019      5.36         6.04     5.30         6.92
east     2015     16.54        18.79    16.52        16.67
         2019      6.09         6.89     5.66         6.99
national 2015     15.80        17.92    16.47        15.73
         2019      5.59         6.44     4.96         6.45
north    2015     15.32        17.50    16.91        15.50
         2019      5.43         6.31     4.83         6.65
south    2015     17.90        20.09    18.71        18.16
         2019      6.24         7.36     5.88         6.88
west     2015     16.89        18.67    18.49        17.20
         2019      5.97         6.68     5.66         7.51

In [11]:
import ctypes
ctypes.CDLL("libgomp.so.1")

import lightgbm
print("LightGBM:", lightgbm.__version__)

LightGBM: 4.7.0


In [8]:
import importlib
import inspect
import hazecast.config as config
import hazecast.train as train

importlib.invalidate_caches()
config = importlib.reload(config)
train = importlib.reload(train)

In [10]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "lightgbm>=4.0"
])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


0

In [12]:
import lightgbm

print(train.__file__)
print(config.MODEL_CANDIDATES)
assert len(config.MODEL_CANDIDATES) == 6

eval_df = train.evaluate_all(gold)

display(
    eval_df.loc[eval_df["split"] == "validation_pooled"]
    .pivot(
        index="target",
        columns="model",
        values=["mae", "rmse", "f2"],
    )
    .round(3)
)

/app/src/hazecast/train.py
('persistence', 'ridge', 'random_forest', 'xgboost', 'lightgbm', 'xgboost_q85')


mae                                                       \
model    lightgbm persistence random_forest  ridge xgboost xgboost_q85   
target                                                                   
central     2.984       3.189         3.005  3.227   2.976       4.222   
east        3.082       3.293         3.175  3.426   3.050       4.662   
national    2.569       2.738         2.693  2.961   2.559       4.239   
north       2.986       2.834         3.132  3.313   2.979       5.086   
south       3.077       3.062         3.129  3.413   3.044       4.830   
west        2.862       2.872         3.108  3.426   2.884       5.314   

             rmse                                                       \
model    lightgbm persistence random_forest  ridge xgboost xgboost_q85   
target                                                                   
central     4.059       4.476         4.065  4.251   4.026       5.221   
east        4.269       4.753         4.400  4.567   4.211       5.719   
national    3.515       3.943         3.738  3.877   3.471       5.146   
north       3.902       4.070         4.138  4.185   3.814       5.986   
south       4.283       4.523         4.326  4.591   4.168       5.800   
west        3.870       4.136         4.225  4.376   3.832       6.307   

               f2                                                       
model    lightgbm persistence random_forest  ridge xgboost xgboost_q85  
target                                                                  
central     0.000       0.000         0.294  0.000   0.000       0.294  
east        0.400       0.200         0.517  0.208   0.000       0.185  
national    0.455       0.250         0.556  0.435   0.263       0.217  
north       0.385       0.000         0.526  0.417   0.500       0.625  
south       0.556       0.429         0.750  0.658   0.455       0.658  
west        0.417       0.250         0.667  0.600   0.217       0.556

In [15]:
# Same table, but for the metric that actually matters for this use case:
# recall on elevated (>=56 ug/m3) days. A model can have a great average
# MAE while still missing most of the days you'd actually want a warning for.
#recall_pivot = eval_df.pivot_table(index=["target", "season"], columns="model", values="recall_elevated")
#recall_pivot.round(2)

KeyError: 'recall_elevated'

In [17]:
import importlib
import hazecast.train as train

importlib.invalidate_caches()
train = importlib.reload(train)

print("Loaded:", train.__file__)
assert hasattr(train, "_log_selected_model")

Loaded: /app/src/hazecast/train.py


In [18]:
# Fits the final XGBoost model per region on ALL available labelled rows
# (not just the LOSO folds) and saves it to data/models/ -- this is what
# score_latest() below actually loads.
model_paths = train.train_final_models(gold)
model_paths

2026/10/22 18:24:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/22 18:24:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/22 18:24:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

{'north': PosixPath('/app/data/models/daily_north_269e53732c3c48818e30784318dbffbf.joblib'),
 'south': PosixPath('/app/data/models/daily_south_000d6f99ae134f3aa28932dda57d8dc0.joblib'),
 'east': PosixPath('/app/data/models/daily_east_306f49d491a5426aa09fcc7ed93e6c55.joblib'),
 'west': PosixPath('/app/data/models/daily_west_7d63fa4e275241de94c1a95529261878.joblib'),
 'central': PosixPath('/app/data/models/daily_central_bb57d37b3b234a31b04fdeb2829b309d.joblib'),
 'national': PosixPath('/app/data/models/daily_national_c3bbf60852d142ba824b4fceb16a59cd.joblib')}

## 5. Score the latest available day

In [19]:
from hazecast.score import score_latest, write_forecast

forecast = score_latest()
write_forecast(forecast)
forecast

,scored_at,as_of_date,forecast_date,region,forecast_pm25,elevated_flag
0,2026-10-22T18:24:48.076469+00:00,2026-10-16,2026-10-17,north,88.8,True
1,2026-10-22T18:24:48.076469+00:00,2026-10-16,2026-10-17,south,108.3,True
2,2026-10-22T18:24:48.076469+00:00,2026-10-16,2026-10-17,east,119.1,True
3,2026-10-22T18:24:48.076469+00:00,2026-10-16,2026-10-17,west,141.1,True
4,2026-10-22T18:24:48.076469+00:00,2026-10-16,2026-10-17,central,130.0,True
5,2026-10-22T18:24:48.076469+00:00,2026-10-16,2026-10-17,national,115.6,True


## Smoke tests (optional)

Fast tests on synthetic fixtures — no real data needed. Good to re-run after any code change to `bronze_to_silver.py` or `silver_to_gold.py` in particular, since these guard the leak-free label construction.

In [20]:
import subprocess, os
env = {**os.environ, "PYTHONPATH": src_path}
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v"],
    cwd=repo_root, capture_output=True, text=True, env=env,
)
print(result.stdout)
print(result.stderr)

============================= test session starts ==============================
platform linux -- Python 3.12.15, pytest-9.1.1, pluggy-1.7.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /app
plugins: anyio-4.15.1, platformdirs-4.13.5
collecting ... collected 3 items

tests/test_pipeline_smoke.py::test_pm25_silver_fills_full_hourly_grid PASSED [ 33%]
tests/test_pipeline_smoke.py::test_firms_daily_silver_flags_long_zero_runs PASSED [ 66%]
tests/test_pipeline_smoke.py::test_gold_labels_are_leak_free_and_shifted_by_one_day PASSED [100%]

============================== 3 passed in 0.22s ===============================


